In [2]:
import Pkg
Pkg.add("Polynomials")

    Updating registry at `C:\Users\dmitr\.julia\registries\General.toml`
   Resolving package versions...
   Installed Setfield ───────── v1.2.0
   Installed Polynomials ────── v4.1.6
   Installed ConstructionBase ─ v1.6.1
   Installed StaticArraysCore ─ v1.4.5
    Updating `C:\Users\dmitr\.julia\environments\v1.12\Project.toml`
  [f27b6e38] + Polynomials v4.1.6
    Updating `C:\Users\dmitr\.julia\environments\v1.12\Manifest.toml`
  [187b0558] + ConstructionBase v1.6.1
  [f27b6e38] + Polynomials v4.1.6
  [efcf1570] + Setfield v1.2.0
  [1e83bf80] + StaticArraysCore v1.4.5
Precompiling packages...
  12966.7 ms  ✓ StaticArraysCore
  12534.9 ms  ✓ Xorg_libICE_jll
  12473.5 ms  ✓ Xorg_libXau_jll
  11463.6 ms  ✓ Xorg_libXdmcp_jll
  12902.8 ms  ✓ EpollShim_jll
  13237.1 ms  ✓ ConstructionBase
  13015.1 ms  ✓ Libmount_jll
  12663.8 ms  ✓ Xorg_libpciaccess_jll
  11563.7 ms  ✓ mtdev_jll
  10691.3 ms  ✓ libevdev_jll
  10742.6 ms  ✓ Xorg_xtrans_jll
   1062.4 ms  ✓ Dbus_jll
   1031.0 ms  ✓ Xorg_xcb

In [18]:
using Polynomials

A = [1.0 2.0 3.0;
     2.0 4.0 5.0;
     3.0 5.0 6.0]

n = size(A, 1)

# n = 3
# A = zeros(n, n)

# for i in 1:n
#     for j in i:n
#         value = round(1 + 9 * rand(), digits=2)
#         A[i,j] = value
#         A[j,i] = value
#     end
# end

println("A =")
display(A)

A =


3×3 Matrix{Float64}:
 1.0  2.0  3.0
 2.0  4.0  5.0
 3.0  5.0  6.0

In [21]:
E = zeros(n, n)
for i in 1:n
    E[i,i] = 1.0
end

D = copy(A)
B_total = copy(E)

for k in n:-1:2
    # step = n - k + 1
    B = copy(E)
    B_inv = copy(E)
    for j in 1:n
        if j == k-1
            B[k-1,j] = 1.0 / D[k,k-1]
        else
            B[k-1,j] = -D[k,j] / D[k,k-1]
        end
        B_inv[k-1,j] = D[k,j]
    end
    D = B_inv * D * B
    B_total = B_total * B
    # println("B$step=")
    # display(B)
    # println("B$step^(-1)=")
    # display(B_inv)
    # println("D$step=")
    # display(D)
end

P = D
p = collect(P[1,:])
println("\nМатрица Фробениуса:")
display(P)

println("\nКоэффициенты:")
for i in 1:n
    println("p$i=", p[i])
end

poly = Polynomial(vcat(-reverse(p), 1.0), :lambda)
println("\nХарактеристический многочлен:")
println(poly)

lambda = sort(real.(roots(poly)))
println("\nСобственные значения: ", round.(lambda, digits=2))

println("\nB =")
display(B_total)

x = Vector{Vector{Float64}}()

for i in 1:n
    yi = [lambda[i]^(n-j) for j in 1:n]
    xi = B_total * yi
    push!(x, xi)
    println("\nlambda$i=", lambda[i])
    println("y$i=", yi)
    println("x$i=", xi)
end

println("\nПроверка ортогональности:")

for i in 1:n-1
    for j in i+1:n
        s = sum(x[i][k] * x[j][k] for k in 1:n)
        println("(x$i,x$j)=", s)
    end
end


Матрица Фробениуса:


3×3 Matrix{Float64}:
 11.0  4.0  -1.0
  1.0  0.0   0.0
  0.0  1.0   0.0


Коэффициенты:
p1=11.000000000000002
p2=3.999999999999986
p3=-1.000000000000007

Характеристический многочлен:
1.0 - 4.0*lambda - 11.0*lambda^2 + 1.0*lambda^3

Собственные значения: [-0.52, 0.17, 11.34]

B =


3×3 Matrix{Float64}:
 -0.384615   4.30769   1.07692
  0.230769  -2.38462  -1.84615
  0.0        0.0       1.0


lambda1=-0.5157294715892574
y1=[0.26597688786573465, -0.5157294715892574, 1.0]
x1=[-1.246979603717469, -0.5549581320873702, 1.0]

lambda2=0.17091518882718063
y2=[0.02921200177183081, 0.17091518882718063, 1.0]
x2=[1.8019377358048412, -2.2469796037174685, 1.0]

lambda3=11.34481428276208
y3=[128.7048111103625, 11.34481428276208, 1.0]
x3=[0.4450418679126167, 0.8019377358048503, 1.0]

Проверка ортогональности:
(x1,x2)=-8.659739592076221e-15
(x1,x3)=8.43769498715119e-15
(x2,x3)=-4.884981308350689e-14


In [20]:
y0 = zeros(n)
y0[1] = 1.0
y = [y0]

for k in 1:n
    push!(y, A * y[end])
end

for k in 0:n
    println("y$k=", y[k+1])
end

M = hcat(y[n:-1:1]...)
C = hcat(M, y[n+1])

println("\nСЛАУ:")
display(C)

for k in 1:n-1
    for i in k+1:n
        factor = C[i,k] / C[k,k]
        for j in k:n+1
            C[i,j] -= factor * C[k,j]
        end
    end
end

p = zeros(n)

for i in n:-1:1
    p[i] = C[i,n+1]
    for j in i+1:n
        p[i] -= C[i,j] * p[j]
    end
    p[i] /= C[i,i]
end

println("\nКоэффициенты:")
for i in 1:n
    println("p$i=", p[i])
end

poly = Polynomial(vcat(-reverse(p), 1.0), :lambda)
println("\nХарактеристический многочлен:")
println(poly)

lambda = sort(real.(roots(poly)))
println("\nСобственные значения:", round.(lambda, digits=2))

x = Vector{Vector{Float64}}()

for i in 1:n
    q = zeros(n)
    q[1] = 1.0
    for j in 2:n
        q[j] = lambda[i] * q[j-1] - p[j-1]
    end
    xi = zeros(n)
    for j in 1:n
        xi += q[j] * y[n-j+1]
    end
    push!(x, xi)
    println("\nlambda$i=", lambda[i])
    println("q$i=", q)
    println("x$i=", xi)
end

println("\nПроверка ортогональности:")

for i in 1:n-1
    for j in i+1:n
        s = sum(x[i][k] * x[j][k] for k in 1:n)
        println("(x$i,x$j)=", s)
    end
end


y0=[1.0, 0.0, 0.0]
y1=[1.0, 2.0, 3.0]
y2=[14.0, 25.0, 31.0]
y3=[157.0, 283.0, 353.0]

СЛАУ:


3×4 Matrix{Float64}:
 14.0  1.0  1.0  157.0
 25.0  2.0  0.0  283.0
 31.0  3.0  0.0  353.0


Коэффициенты:
p1=10.999999999999996
p2=4.000000000000031
p3=-0.9999999999999782

Характеристический многочлен:
1.0 - 4.0*lambda - 11.0*lambda^2 + 1.0*lambda^3

Собственные значения:[-0.52, 0.17, 11.34]

lambda1=-0.5157294715892565
q1=[1.0, -11.515729471589253, 1.939001075347523]
x1=[4.42327160375827, 1.9685410568214934, -3.54718841476776]

lambda2=0.1709151888271759
q2=[1.0, -10.829084811172821, -5.850855075327137]
x2=[-2.679939886499958, 3.3418303776543574, -1.4872544335184656]

lambda3=11.344814282762076
q3=[1.0, 0.3448142827620799, -0.0881460000204255]
x3=[14.256668282741654, 25.68962856552416, 32.03444284828624]

Проверка ортогональности:
(x1,x2)=1.4299672557172016e-13
(x1,x3)=1.4210854715202004e-14
(x2,x3)=-6.394884621840902e-14
